# Project 3

### Imports & Constants

In [1]:
from vpython import *
import matplotlib.pyplot as plt
%matplotlib inline
G = 6.6743 * 10**-20 #km^3*kg^-1*s^-2

C:\Users\lucas\anaconda3\envs\lab_0\lib\site-packages\vpython\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


<IPython.core.display.Javascript object>

## Newton's Cannonball

#### assuming:

In [2]:
def newton(velocity):
    #objects
    earth = sphere(pos=vector(0,0,0), radius=6371, color=color.blue)
    ball = sphere(pos=vector(0,6471,0), radius=100, color=color.red, make_trail=True)

    #constants
    delta_t = 0.01
    time = 0
    F = 0
    r = 0
    m = 5.97217 * 10**24 #kg
    v_x = velocity
    v_y = 0

    moving = 1
    while(moving):
        rate(25000)
        
        #gravitational force equation
        r = sqrt(ball.pos.x**2 + ball.pos.y**2)
        F = -G*m/r**2
        r_x = ball.pos.x / r
        r_y = ball.pos.y / r

        #update velocities
        v_x += F*r_x*delta_t
        v_y += F*r_y*delta_t
        
        #update positions
        ball.pos.x += v_x*delta_t
        ball.pos.y += v_y*delta_t

        time += delta_t

        #stops the simulation either when the ball returns to its initial position or after 30 seconds
        if (time > 100 and ball.pos.x >= 0 and ball.pos.x <= 1 and ball.pos.y >= 100 or r <= 6371 or time > 7500):
            moving = 0

### Orbital velocity

#### formula

#### simulation

In [3]:
scene = canvas()
scene.camera.pos = vector(-330,0,0)
scene.camera.axis = vector(-0.67, -1.18, -1.07)
scene.autoscale = True

<IPython.core.display.Javascript object>

In [4]:
newton(7.848)

### Escape velocity

#### formula

In [5]:
scene = canvas()
scene.camera.pos = vector(-330,0,0)
scene.camera.axis = vector(-0.67, -1.18, -1.07)
scene.autoscale = True

<IPython.core.display.Javascript object>

In [6]:
newton(11.1)

### Resources

## Solar System

### Planet class

In [7]:
class Planet:

    def __init__(self, radius, mass, x, y, z): #radius is just for simulation scaling
        self.radius = radius
        self.mass = mass
        self.ogx = x #store the original coords for future usage (calcualting one year)
        self.ogy = y
        self.ogz = z
        self.x = x #these coords are used by the actual vpython object to read its location
        self.y = y
        self.z = z

    def set_position(self, x, y, z): #km
        self.x = x
        self.y = y
        self.z = z

    def set_velocity(self, xv, yv, zv): #km/s
        self.xv = xv
        self.yv = yv
        self.zv = zv

    def new_position(self, x, y, z): #t+1 for position; temporarily stored here before update is called to actually move each planet
        self.new_x = x
        self.new_y = y
        self.new_z = z

    def new_velocity(self, xv, yv, zv): #same idea as above
        self.new_xv = xv
        self.new_yv = yv
        self.new_zv = zv

    def reset_location(self): #resets the planet to its original location for multiple simulation runs
        self.x = self.ogx
        self.y = self.ogy
        self.z = self.ogz

    def calculations(self, all_planets, delta_t):
        #sun calculations: distance, force, then direction and acceleration applied to new velocity and position values
        dx, dy, dz = sun.x-self.x, sun.y-self.y, sun.z-self.z #calculate distance pointing from target planet to sun
        r = sqrt(dx**2+dy**2+dz**2) #vectorize
        F = G*sun.mass*self.mass/r**2 #calculate force towards the sun
        r_x, r_y, r_z = dx/r, dy/r, dz/r #directional unit vectors
        F_x, F_y, F_z = F*r_x, F*r_y, F*r_z #directional force vectors

        #repeat above process but for all planets
        for planet in all_planets:
            if (planet.radius == self.radius):
                pass
            else:
                dx, dy, dz = planet.x-self.x, planet.y-self.y, planet.z-self.z
                r = sqrt(dx**2+dy**2+dz**2)
                F = G*planet.mass*self.mass/r**2
                r_x, r_y, r_z = dx/r, dy/r, dz/r
                F_x += F*r_x
                F_y += F*r_y
                F_z += F*r_z

        #sets the new velocity and position with respect to time and mass (force to acceleration)
        self.new_velocity(self.xv + F_x/self.mass*delta_t, self.yv + F_y/self.mass*delta_t, self.zv + F_z/self.mass*delta_t)
        self.new_position(self.x + self.new_xv*delta_t, self.y + self.new_yv*delta_t, self.z + self.new_zv*delta_t)

    def update(self): #the function that is called to actually move each planet (keep states consistent)
        self.set_velocity(self.new_xv, self.new_yv, self.new_zv)
        self.set_position(self.new_x, self.new_y, self.new_z)
        

### Planet assignments

In [8]:
#create the sun, which can be accessed globally
sun = Planet(695700, 1988410*10**24, 0,0,0) #km, kg

#create every planet and set their initial position in the initialization and then set their velocity
mercury = Planet(2439.4, 3.302*10**23, -3.252036351334026e7, 3.682184140644412e7, 5.993726922189429e6)
mercury.set_velocity(-4.637170192071822e1, -3.029593266151278e1, 1.785451406333900e0)

venus = Planet(6051.84, 48.685*10**23, 6.764183600427456e7, -8.524859720375101e7, -5.064590323585507e6)
venus.set_velocity(2.720041308002972e1, 2.164358109200013e1, -1.276345693883823e0)

earth = Planet(6371.01, 5.97219*10**24, -8.018612039327677e7, -1.283639821608722e8, -8.278023510448635e3)
earth.set_velocity(2.478456603258307e1, -1.590470745685047e1, -1.774036396904322e-4)

mars = Planet(3389.92, 6.4171*10**23, -1.759191371051778e8, 1.740224514710827e8, 7.977539613769904e6)
mars.set_velocity(-1.612015375369448e1, -1.516597828061472e1, 7.947835462549246e-2)

#create a parsible list for all planets
all_planets = [mercury, venus, earth, mars]

### Simulation

In [9]:
scene = canvas()

<IPython.core.display.Javascript object>

#### assumptions/notes

In [10]:
sun_scalar = 0.5*10**5
planet_scalar = 0.8*10**3
distance_scalar = 0.9*10**6

In [11]:
def run_sim(delta_t, show):

    #reset locations and velocities if necessary
    for planet in all_planets:
        planet.reset_location()
    mercury.set_velocity(-4.637170192071822e1, -3.029593266151278e1, 1.785451406333900e0)
    venus.set_velocity(2.720041308002972e1, 2.164358109200013e1, -1.276345693883823e0)
    earth.set_velocity(2.478456603258307e1, -1.590470745685047e1, -1.774036396904322e-4)
    mars.set_velocity(-1.612015375369448e1, -1.516597828061472e1, 7.947835462549246e-2)

    #create the vpython object, all scaled with respect to the scalars created in the above cell
    sun_sim = sphere(pos=vector(0,0,0), radius=sun.radius/sun_scalar, color=color.yellow)
    mercury_sim = sphere(pos=vector(mercury.x/distance_scalar, mercury.y/distance_scalar, mercury.z/distance_scalar), radius=mercury.radius/planet_scalar, color=color.magenta, make_trail=True)
    venus_sim = sphere(pos=vector(venus.x/distance_scalar, venus.y/distance_scalar, venus.z/distance_scalar), radius=venus.radius/planet_scalar, color=color.red, make_trail=True)
    earth_sim = sphere(pos=vector(earth.x/distance_scalar, earth.y/distance_scalar, earth.z/distance_scalar), radius=earth.radius/planet_scalar, color=color.green, make_trail=True)
    mars_sim = sphere(pos=vector(mars.x/distance_scalar, mars.y/distance_scalar, mars.z/distance_scalar), radius=mars.radius/planet_scalar, color=color.orange, make_trail=True)

    #for optimization; allows for less checks in the loop; see year checks below
    planets_done = [0,0,0,0]
    year_lengths = [0,0,0,0] #in days
    
    time = 0
    sim = 1
    
    while(sim):
        if(show):
            rate(10000)

        #perform each calculation which is stored temporarily and THEN update all planet simulataneously
        [planet.calculations(all_planets, delta_t) for planet in all_planets]
        [planet.update() for planet in all_planets]

        #update vpython objects to match the Planet objects above
        mercury_sim.pos = vector(mercury.x/distance_scalar, mercury.y/distance_scalar, mercury.z/distance_scalar)
        venus_sim.pos = vector(venus.x/distance_scalar, venus.y/distance_scalar, venus.z/distance_scalar)
        earth_sim.pos = vector(earth.x/distance_scalar, earth.y/distance_scalar, earth.z/distance_scalar)
        mars_sim.pos = vector(mars.x/distance_scalar, mars.y/distance_scalar, mars.z/distance_scalar)

        #add elapsed time per loop to total time
        time += delta_t

        ###YEAR CHECKS
        #below are checks for year completions
        #the checks only start relatively close to the expected day to ensure less processor load per loop
        #the checks also cease once that specific planet's year is done, hence the need for a planets_done list
        #this also ensures that a planet's year is not overridden upon its next orbit completion
        if (time/60/60/24 > 70 and not planets_done[0] and sqrt((mercury.x-mercury.ogx)**2+(mercury.y-mercury.ogy)**2) < 100000):
            planets_done[0] = 1
            year_lengths[0] = time/60/60/24
        if (time/60/60/24 > 200 and not planets_done[1] and sqrt((venus.x-venus.ogx)**2+(venus.y-venus.ogy)**2) < 100000):
            planets_done[1] = 1
            year_lengths[1] = time/60/60/24
        if (time/60/60/24 > 350 and not planets_done[2] and sqrt((earth.x-earth.ogx)**2+(earth.y-earth.ogy)**2) < 100000):
            planets_done[2] = 1
            year_lengths[2] = time/60/60/24
        if (time/60/60/24 > 650 and not planets_done[3] and sqrt((mars.x-mars.ogx)**2+(mars.y-mars.ogy)**2) < 100000):
            year_lengths[3] = time/60/60/24
            return year_lengths

In [12]:
year_lengths = run_sim(50, True)

### Discussion

### c.

### d.

### e.

In [13]:
print(f'Mercury: {year_lengths[0]}\nVenus: {year_lengths[1]}\nEarth: {year_lengths[2]}\nMars: {year_lengths[3]}')

Mercury: 87.94733796296298
Venus: 224.66608796296296
Earth: 365.625
Mars: 686.9357638888888


### f.

In [ ]:
delta_t_set = [50, 250, 1000, 5000]

set_1 = run_sim(delta_t_set[0], False)
set_2 = run_sim(delta_t_set[1], False)
set_3 = run_sim(delta_t_set[2], False)
set_4 = run_sim(delta_t_set[3], False)

mercury_year_error = [abs(x - 88) for x in [set_1[0], set_2[0], set_3[0], set_4[0]]]
venus_year_error   = [abs(x - 224.7) for x in [set_1[1], set_2[1], set_3[1], set_4[1]]]
earth_year_error   = [abs(x - 365.25) for x in [set_1[2], set_2[2], set_3[2], set_4[2]]]
mars_year_error    = [abs(x - 687) for x in [set_1[3], set_2[3], set_3[3], set_4[3]]]

plt.plot(delta_t_set, mercury_year_error, 'o-', label="Mercury")
plt.plot(delta_t_set, venus_year_error, 'o-', label="Venus")
plt.plot(delta_t_set, earth_year_error, 'o-', label="Earth")
plt.plot(delta_t_set, mars_year_error, 'o-', label="Mars")

plt.title("Time step vs. Planet year error")
plt.xlabel("Time step (seconds per step)")
plt.ylabel("Planet year error (in days)")
plt.legend()
plt.show()